# ADAC Workshop 2 (26L) — Distributed SparkML on Managed Service for Apache Spark

You run a **real distributed machine-learning pipeline** on the Microsoft Security Incident (GUIDE) dataset from Lab 1 — this time on **Google Managed Service for Apache Spark** (Dataproc Serverless, Lightning engine, 2 executors), driven from a **Vertex AI Workbench** notebook that **you provision yourself**, in **your own** GCP project funded by your education coupon.

**The ML pipeline you build here is the same pattern you already saw in the DS course** — your reference notebook is [`ds-notebooks/session_2/ml.ipynb`](https://github.com/biodatageeks/ds-notebooks/blob/main/session_2/ml.ipynb) (`StringIndexer → OneHotEncoder → VectorAssembler → classifier → CrossValidator`). There it ran on a small table on a single local Spark. Here you scale that exact pattern to **~9.5 M rows** across a distributed cluster, and watch *how it scales*.

#### Student tasks

- Sections tagged **Student task N** are the parts you write, run, and interpret yourself.
- Your reference model for the ML parts is the DS-course notebook `ml.ipynb`.

#### Cost

- Everything runs in **your coupon project** and bills you, specifically:
  - the Vertex AI Workbench VM,
  - the Google Cloud Storage (GCS) bucket,
  - the Managed Service for Apache Spark compute.
- Keep cost down: **use 2 executors**.
- **Tear everything down at the end** (Part 4.3).


## Part 0 — Provision your own infrastructure

You set up the infrastructure yourself: the Vertex AI Workbench, the Google Cloud Storage (GCS) bucket, and the Managed Service for Apache Spark session in your own project, then connect them.

Follow the commands and console steps in order. In 0.4, two `gsutil` lines are left for you to complete.

### 0.1 Launch your Vertex AI Workbench instance

You will run **this very notebook** from a Vertex AI Workbench instance (managed JupyterLab on a GCP VM), not from Colab. Provision it once in the console:

**Console** ([console.cloud.google.com/vertex-ai/workbench](https://console.cloud.google.com/vertex-ai/workbench/instances)):
1. **Vertex AI → Workbench → Instances** → **Create new**.
2. **Name** `adac-lab2` (lowercase letters/digits/hyphens), **Region** `europe-west1`.
3. Leave the defaults — the machine type `e2-standard-4` is more than enough for this lab. This Vertex AI Workbench VM is **not** the Spark cluster; it only runs JupyterLab and the **Spark Connect client**, which sends your code to the remote session. The Spark **driver and executors** both run on the **Managed Service for Apache Spark** session, not on this VM. Click **Advanced options** only if you want to change the machine type.
4. Leave the **service account** as the default Compute Engine service account (we rely on it for auth — see Part 0.2).
5. **Create**. When the status turns green, click the **Open JupyterLab** link, upload this notebook (or `git clone` it), and run the rest from there.

**gcloud (equivalent):**
```bash
gcloud workbench instances create adac-lab2 \
  --project=YOUR_PROJECT_ID --location=europe-west1-b \
  --machine-type=e2-standard-4
```

### 0.2 Authentication — why no sign-in is required

In Lab 1 (Colab) you authenticated interactively with `from google.colab import auth; auth.authenticate_user()` and a Google sign-in popup. In this lab that step is not needed.

This notebook runs on the Vertex AI Workbench VM, which is itself a resource inside your project. Google client libraries — `gsutil`, the Cloud Storage client, the Spark Connect client — automatically use Application Default Credentials (ADC). On a Workbench VM, ADC resolves to the VM's attached service account. The Vertex AI Workbench VM, the Google Cloud Storage bucket, and the Managed Service for Apache Spark session all run under the same project and the same service account, so requests between them are authorized without any interactive step.

The cell below prints the active identity so you can confirm which account you are running as.

In [ ]:
# No google.colab.auth here — ADC is automatic on a Workbench VM.
# Confirm the active identity (the instance service account) and project:
!gcloud auth list
!gcloud config list project

### 0.3 Get the dataset (KaggleHub, like Lab 1)

Download GUIDE onto the Workbench VM's local disk first; you push it to your bucket in Part 0.4 of this notebook.


In [ ]:
%pip install -q kagglehub google-cloud-storage dataproc-spark-connect


In [ ]:
import kagglehub, glob, os
path = kagglehub.dataset_download('Microsoft/microsoft-security-incident-prediction')
train_csv = glob.glob(os.path.join(path, '**', 'GUIDE_Train.csv'), recursive=True)[0]
test_csv  = glob.glob(os.path.join(path, '**', 'GUIDE_Test.csv'),  recursive=True)[0]
print(train_csv, test_csv)

### 0.4 Create your bucket and upload the data

The bucket workflow has three steps: create a bucket with `gsutil mb`, copy the data into it with `gsutil cp`, then read it later from Spark with `CREATE TABLE … LOCATION 'gs://…'`. This is the standard GCS pattern; the DS-course materials use it as a reference.

The bucket name follows the convention `adac-{semester}-{id}` and lives in your own project. Use `26l` for the semester (this 26L edition) and your Google username for the id. GCS bucket names are globally unique, so using your username avoids clashes with other students' buckets.

> **Student task (mini):** complete the two `gsutil` lines marked `# TODO` — create the bucket in the correct region, and upload the **train** CSV. The **test** CSV upload is provided as a worked example.

In [ ]:
SEMESTER = '26l'            # this 26L edition (bucket names must be lowercase)
USER_ID  = ''               # TODO: your Google username, lowercase (the part before @ in your Google email)
PROJECT_ID = ''             # TODO: your coupon project id
REGION   = 'europe-west1'
BUCKET   = f'adac-{SEMESTER}-{USER_ID}'   # globally-unique bucket name

!gcloud config set project {PROJECT_ID}

# TODO: create the bucket in {REGION}  (hint: gsutil mb -l <region> gs://<bucket>)
!gsutil mb -l {REGION} gs://{BUCKET} || echo 'bucket may already exist'

# worked example — uploading the TEST csv:
!gsutil -m cp '{test_csv}'  gs://{BUCKET}/guide/GUIDE_Test.csv
# TODO: upload the TRAIN csv to gs://{BUCKET}/guide/GUIDE_Train.csv the same way
!gsutil -m cp '{train_csv}' gs://{BUCKET}/guide/GUIDE_Train.csv

TRAIN = f'gs://{BUCKET}/guide/GUIDE_Train.csv'
TEST  = f'gs://{BUCKET}/guide/GUIDE_Test.csv'
print(TRAIN); print(TEST)

### 0.5 Connect from Vertex AI Workbench via Spark Connect

`DataprocSparkSession` resolves the session from project + region + session id and authenticates via **ADC** (the instance service account). There is **no `sc://` endpoint to copy**.


In [ ]:
SESSION_ID = 'adac-lab2'  # auto-created on connect by .getOrCreate() below
from google.cloud.dataproc_spark_connect import DataprocSparkSession
spark = (DataprocSparkSession.builder
         .projectId(PROJECT_ID)
         .location(REGION)
         .dataprocSessionId(SESSION_ID)
         .runtimeVersion('2.3')
         .config('dataproc.tier', 'premium')
         .config('spark.dataproc.engine', 'lightningEngine')
         .config('spark.executor.instances', '2')
         .getOrCreate())
print('Spark version:', spark.version)
print(spark.range(5).count())  # first 'touch Big Data' moment

## Part 1 — Load the data and build the SparkML pipeline

### 1.1 Distributed read + a first look

We register the GCS CSV as a Spark SQL table with `CREATE TABLE … USING csv … LOCATION 'gs://…'` — the same idiom from the DS course — then drop the ~0.5 % of rows with no `IncidentGrade` label.


In [ ]:
from pyspark.sql import functions as F

spark.sql('DROP TABLE IF EXISTS guide_train')
spark.sql(f'''CREATE TABLE guide_train
              USING csv
              OPTIONS (header true, inferSchema true)
              LOCATION "{TRAIN}"''')

raw = spark.table('guide_train').where(F.col('IncidentGrade').isNotNull())
print('rows:', raw.count())
raw.groupBy('IncidentGrade').count().show()

### 1.2 Student task 1 — build the pipeline (Transformers + Estimators)

Following the DS-course `ml.ipynb` pattern, build a SparkML `Pipeline` that turns the raw categorical columns into one feature vector + one label, ending in a classifier:

- one `StringIndexer` **per categorical feature** (`handleInvalid='keep'`),
- a `OneHotEncoder` over those indices,
- a `VectorAssembler` into a single `features` column,
- a `StringIndexer` for the **label** `IncidentGrade` → `label` (`handleInvalid='skip'`),
- a `RandomForestClassifier(featuresCol='features', labelCol='label')`.

Use the feature columns `['Category', 'EntityType', 'EvidenceRole', 'CountryCode']` (chosen in Lab 1 — all 0 % nulls).

> Reminder of the concepts (from `ml.ipynb`): a **Transformer** maps DataFrame→DataFrame (`transform`), an **Estimator** is `fit` to produce a Transformer. `Pipeline.fit()` runs the stages in order and returns a `PipelineModel`. Nothing computes until an action triggers the execution **DAG** across the executors.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

CAT_COLS = ['Category', 'EntityType', 'EvidenceRole', 'CountryCode']

# TODO: index the label IncidentGrade -> 'label'  (StringIndexer, handleInvalid='skip')
# TODO: one StringIndexer per feature in CAT_COLS  (handleInvalid='keep')
# TODO: OneHotEncoder over the indexed columns
# TODO: VectorAssembler -> 'features'
# TODO: RandomForestClassifier(featuresCol='features', labelCol='label')
# TODO: assemble everything into a Pipeline and print the stage types

## Part 2 — Fit, evaluate, tune, and read the distributed execution

### 2.1 Student task 2 — fit, predict, evaluate

Fit the pipeline on **Train**, transform **Test**, print a confusion matrix, and report the model quality.

> ⚠️ **Difference from the `ml.ipynb` template — read carefully.** In the DS course the target was **binary** (`compAboveAvg`), so it used `BinaryClassificationEvaluator` + AUROC, and even tried `GBTClassifier`. **GUIDE's `IncidentGrade` is 3-class**, so you must use **`MulticlassClassificationEvaluator`** (metrics `f1` and `accuracy`) — `BinaryClassificationEvaluator`/AUROC do **not** apply, and Spark's `GBTClassifier` is **binary-only** (it will error on 3 classes). Everything else (the indexer/encoder/assembler pattern, `fit`/`transform`, the confusion matrix) carries over unchanged.

We cache the inputs so fit time is not dominated by re-reading from GCS.


In [ ]:
import time
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# given: cache train/test so fit time isn't dominated by re-reading from GCS
train = raw.cache(); train.count()
test  = (spark.read.option('header', True).option('inferSchema', True).csv(TEST)
         .where(F.col('IncidentGrade').isNotNull()).cache()); test.count()

# TODO: fit the pipeline on `train` (time it) -> model
# TODO: transform `test` -> pred
# TODO: print the confusion matrix: pred.groupBy('label','prediction').count()...
# TODO: evaluate with MulticlassClassificationEvaluator  (metrics 'f1' AND 'accuracy')
#       NOTE: GUIDE is 3-class -> use Multiclass, NOT BinaryClassificationEvaluator/AUROC.
# Keep your baseline F1 in a variable `f1` -- you compare against it in 2.2 and 4.2.

### 2.2 Student task 3 — hyperparameter tuning with CrossValidator

Exactly as in `ml.ipynb` (cells 49–57), tune the model with `ParamGridBuilder` + `CrossValidator`: sweep a couple of `RandomForestClassifier` hyperparameters (e.g. `numTrees` and `maxDepth`), 3-fold CV on Train, pick the `bestModel`, evaluate it on Test, and **compare its F1 to the baseline from 2.1**.

> Note: the evaluator inside the `CrossValidator` must also be a `MulticlassClassificationEvaluator` (same 3-class reason as 2.1). Keep the grid small — every grid point × every fold is a full distributed `fit`, and you are paying for it.


In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# TODO: build a small ParamGridBuilder over rf.numTrees and rf.maxDepth
# TODO: CrossValidator(estimator=pipeline, estimatorParamMaps=grid,
#                      evaluator=MulticlassClassificationEvaluator(... metricName='f1'),
#                      numFolds=3, parallelism=2, seed=42)
# TODO: fit on `train`, pull the bestModel, evaluate on `test`
# TODO: print baseline F1 (from 2.1) vs tuned F1

### 2.3 Read the Spark UI / DAG

Open the **Spark UI** for your session (Dataproc → Serverless → Interactive Sessions → your session → **View Spark UI**). Find the job triggered by the `fit` in 2.1, open its **DAG**, and note:
- how many **stages** the job has,
- where the **shuffle** boundaries are,
- how tasks were split across your **2 executors**.

Record your observations in the answer cells in **Part 4.1**.


## Part 3 — How does it scale?

**Weak scaling:** run the *same* pipeline on growing input and watch wall-clock and throughput (rows/sec). We sweep fractions of Train; a warm-up run excludes catalog/JIT/IO warm-up. Executors stay fixed at **2**.


In [ ]:
import time
_ = pipeline.fit(train.sample(0.05, seed=1))  # warm-up, not measured
fracs = [0.1, 0.25, 0.5, 1.0]
rows, secs = [], []
for fr in fracs:
    d = train.sample(fr, seed=1).cache(); n = d.count()
    t0 = time.perf_counter(); pipeline.fit(d); dt = time.perf_counter()-t0
    d.unpersist()
    rows.append(n); secs.append(dt)
    print(f'frac={fr:>4}  rows={n:>9,}  fit={dt:6.1f}s  thr={n/dt:,.0f} rows/s')

### 3.1 Plot time and throughput vs data volume


In [ ]:
import matplotlib.pyplot as plt
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot(rows, secs, 'o-'); a.set_xlabel('rows'); a.set_ylabel('fit time (s)'); a.set_title('Scaling: time')
thr = [n/s for n, s in zip(rows, secs)]
b.plot(rows, thr, 'o-'); b.set_xlabel('rows'); b.set_ylabel('rows/sec'); b.set_title('Scaling: throughput')
plt.tight_layout(); plt.show()

### 3.2 (Optional bonus) 4 executors — a peek at strong scaling

If your coupon allows: **terminate this session**, create a new one with `spark.executor.instances=4`, re-connect (0.6), and re-run the `frac=1.0` point. Compare fit time to your 2-executor result.


## Part 4 — Conclusions, improvement task, teardown

### 4.1 Your answers  *(part of your submission)*
Fill these in from your Spark UI (2.3) and scaling plot (Part 3).


In [ ]:
from IPython.display import Markdown, display
PROMPT = 'How many Spark stages did `fit` create, and where was the shuffle?'
ANSWER = ''  # TODO: your answer from the Spark UI (2.3)
display(Markdown('**' + PROMPT + '**\n\n' + ANSWER))

In [ ]:
from IPython.display import Markdown, display
PROMPT = 'From your scaling plot: is throughput (rows/sec) roughly flat as data grows? What does that imply?'
ANSWER = ''  # TODO: your interpretation of the Part 3 plot
display(Markdown('**' + PROMPT + '**\n\n' + ANSWER))

In [ ]:
from IPython.display import Markdown, display
PROMPT = 'What did the 2 executors give you vs single-node pandas in Lab 1 and the local Spark in the DS course ml.ipynb?'
ANSWER = ''  # TODO: your answer
display(Markdown('**' + PROMPT + '**\n\n' + ANSWER))

### 4.2 Student task 4 — improve the model (open-ended)

This is the open-ended task from the end of `ml.ipynb` ("Czy można jeszcze poprawić jakość predykcji?"), now on GUIDE. **Beat your baseline F1 from 2.1** by at least one of:

- **(a) adding a feature** — bring another low-null categorical/numeric column into the pipeline,
- **(b) changing the model** — e.g. `LogisticRegression` or `MultilayerPerceptronClassifier` (both support multiclass; remember GBT does not),
- **(c) better hyperparameters** — extend the grid from 2.2.

Report your best F1 and what moved it.


In [ ]:
# TODO (open-ended): beat your baseline F1 from 2.1 by one of:
#   (a) adding a low-null feature to CAT_COLS / the assembler,
#   (b) switching the classifier (LogisticRegression / MultilayerPerceptronClassifier
#       both support multiclass; GBTClassifier does NOT),
#   (c) widening the hyperparameter grid from 2.2.
# Report your best F1 and what moved it.

### 4.3 Tear everything down — *save your coupon!*

You provisioned three billable things; tear down **all three**. Idle Vertex AI Workbench VMs and live Managed Service for Apache Spark sessions keep charging.


In [ ]:
# 1. Stop the Spark session (releases the serverless executors)
try:
    spark.stop()
except Exception as e:
    print('stop:', e)

# 2. Terminate the Managed Service for Apache Spark session
!gcloud beta dataproc sessions terminate {SESSION_ID} --location={REGION} --project={PROJECT_ID} --quiet || true

# 3. (Optional) delete the bucket and data
# !gsutil -m rm -r gs://{BUCKET}

# 4. Stop or delete your Vertex AI Workbench instance (do this from the console too):
# !gcloud workbench instances stop adac-lab2 --location=europe-west1-b
# !gcloud workbench instances delete adac-lab2 --location=europe-west1-b --quiet
print('Done. Verify in the console that the session is TERMINATED and the Workbench is STOPPED/DELETED.')